In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["KERAS_BACKEND"] = "torch"

import util.supabase_utils as su
import src.config as config
import src.data_preprocessing.text_preprocessing as text_pre
from src.data_preprocessing.TextEmbedder import TextEmbedder
from src.agents.util.Retriever import Retriever
from src.agents.util.ResponseModel import ResponseModel
from agents.NLP_Agent import NLPAgent
from src.agents.Visual_Agent import VisualModel
import src.data_preprocessing.image_preprocessing as img_pre

2026-10-07 19:17:14.103303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791415034.113316    5804 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791415034.116243    5804 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791415034.124015    5804 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791415034.124027    5804 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791415034.124028    5804 computation_placer.cc:177] computation placer alr

In [2]:

plant_expert_vqa_TRAIN= text_pre.load_csv(config.Data.PlantExpertVQA.TRAIN_FILE)
plant_expert_vqa_TEST= text_pre.load_csv(config.Data.PlantExpertVQA.TEST_FILE)
plant_expert_vqa_VAL= text_pre.load_csv(config.Data.PlantExpertVQA.VALIDATION_FILE)

# training

text_pre.preprocess_dataframe(
    plant_expert_vqa_TRAIN,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

# testing
text_pre.preprocess_dataframe(
    plant_expert_vqa_TEST,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

#validation
text_pre.preprocess_dataframe(
    plant_expert_vqa_VAL,
    config.Data.PlantExpertVQA.TEXT_COLUMNS,
    config.Data.PlantExpertVQA.COLUMNS_TO_REMOVE,
    config.Data.PlantExpertVQA.NA_FILL,
    ["image_path"],
    config.Data.PlantExpertVQA.ROOT,
)

19:17:17 [DEBUG   ]: [load_csv] Successfully loaded /mnt/c/Users/codyr/Code/AI-572/PlantQA/data/PlantExpertVQA/data/train.csv into dataframe


/mnt/c/Users/codyr/Code/AI-572/PlantQA/src/data_preprocessing/text_preprocessing.py:44: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  data= pd.read_csv(csv_path)


19:17:17 [DEBUG   ]: [load_csv] Successfully loaded /mnt/c/Users/codyr/Code/AI-572/PlantQA/data/PlantExpertVQA/data/test.csv into dataframe
19:17:18 [DEBUG   ]: [load_csv] Successfully loaded /mnt/c/Users/codyr/Code/AI-572/PlantQA/data/PlantExpertVQA/data/val.csv into dataframe


In [3]:
embedder = TextEmbedder("sentence-transformers/all-MiniLM-L6-v2")
client = su.connect_supabase()
retriever = Retriever(client, embedder)

retriever.retrieve("how do I treat leaf spot")[:1]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[{'id': 2772,
  'title': 'AMCA4',
  'body': 'be infected by foliar diseases like leaf spot, rust, or powdery mildew. To help prevent leaf spot, fallen leaves should be raked away and overhead irrigation should be avoided. Pests may include Japanese beetles, spider mites, aphids, and leaf miners (Dickert, 2010). The',
  'plant_code': 'AMCA4',
  'common_name': 'Canadian serviceberry',
  'scientific_name': 'Amelanchier canadensis (L.) Medik.',
  'genus': 'Amelanchier Medik.',
  'similarity': 0.63679051399231}]

In [4]:
train_sample = (plant_expert_vqa_TRAIN.drop_duplicates(subset=["question_text", "answer"]).sample(20000, random_state=0))

In [5]:
model = ResponseModel(lora=True)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [6]:
model.train_model(train_sample, retriever=None, epochs=1,
                  out="./models/response_models/10_6")

Step,Training Loss
500,1.433674
1000,0.913901


In [5]:
visual_model = VisualModel.load("./models/visual_models/10_6.keras",
                                "./models/visual_models/10_6_classes.json")
response_model = ResponseModel(adapter="./models/response_models/10_6")
agent = NLPAgent(retriever=retriever, response_model=response_model, visual_model=visual_model)

for row in plant_expert_vqa_VAL.sample(3, random_state=0).itertuples():
    image = img_pre.load_image(row.image_path)
    answer = agent.ask_question(image=image, question=row.question_text)

    print(f"question:    {row.question_text}")
    print(f"pred_answer: {answer[:180]}\n")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

question:    raspberry leaf
pred_answer: no

question:    what disease shown here
pred_answer: yes

question:    lead through leaf related observations photograph
pred_answer: severe late blight tomato leaf spots target pattern dark brown black border

